# Follow-up experiments for paper 2: cell B (Colab, T4)

Runs the remaining experiments of **cell B** (CIFAR-10 × BN CNN), because the benchmark runs of cell B were made on Colab:
1. **E5**: seeds 3, 4 for the weight-decay values **next to the best one** (`adam-x0.001`, `adam-x0.1`, `adamw-x0.1`, `adamw-x10`).
2. **E6**: check of $\rho_\ell$ against the actual update (`rel_upd`, `ang_upd`, `rho_step`) for the 5 configurations of Section V-C (`adam`, `adam-x0.01`, `adamw`, `sfadamw-l2`, `sfadamw`, seeds 0–1).
3. **E5b**: seeds 3, 4 for the remaining weight-decay values (`adam-x10`, `adam-x100`, `adamw-x0.01`), so that the whole of Table IV / Fig. 1 has 5 seeds.

About **2.0 T4 hours** (one Colab session). Cells C (ResNet-18) and E run in parallel on Kaggle (`kaggle_exp.ipynb`).

- Code: upload `adam-bench-v4.zip` to `MyDrive` (the notebook unzips it), or upload the folder as `MyDrive/adam-bench-v4`.
- Results are written to `MyDrive/adam-bench-v4/results_exp`, so a lost session loses no data.
- Runtime → Change runtime type → **T4 GPU**.
- If the session is interrupted: reopen it and **Runtime → Run all**. Finished runs are skipped; an unfinished run resumes from its checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/adam-bench-v4'   # change if stored elsewhere
RESULTS = f'{PROJECT}/results_exp'
WORK = '/content/adam-bench-v4'
DATA = '/content/data'
CELLS = ['B']
EXPS = ['E5', 'E6', 'E5b']           # in order of priority
RUN_SELFTEST = True                  # True the first time; set to False once the selftest has passed

import os, zipfile
ZIP = '/content/drive/MyDrive/adam-bench-v4.zip'
if os.path.exists(ZIP):   # always unzip the latest code (the zip holds no results, so nothing is overwritten)
    zipfile.ZipFile(ZIP).extractall('/content/drive/MyDrive')
    print('Unzipped', ZIP)
assert os.path.exists(f'{PROJECT}/run_exp.py'), f'No code found in {PROJECT}'
!rsync -a --delete --exclude results_exp --exclude data {PROJECT}/ {WORK}/
%cd {WORK}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# === Use the torch + cuDNN of the benchmark (the runner is a subprocess, so NO restart is needed) ===
import subprocess, sys
WANT = {'B': ('2.11.0', '0.26.0', 'cu130', 92700),
        'C': ('2.11.0', '0.26.0', 'cu128', 91900),
        'D': ('2.11.0', '0.26.0', 'cu128', 91900),
        'E': ('2.11.0', '0.26.0', 'cu128', 91900)}
tv, vv, cu, cudnn_want = WANT[CELLS[0]]
want = f'{tv}+{cu}'

def probe():
    out = subprocess.run([sys.executable, '-c',
                          'import torch; print(torch.__version__); print(torch.backends.cudnn.version()); '
                          'print("|".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())))'],
                         capture_output=True, text=True).stdout.splitlines()
    return out[0], int(out[1]), out[2] if len(out) > 2 else ''

have, cudnn, gpus = probe()
print('torch installed:', have, '| needed:', want)
if have != want:
    print('Reinstalling torch to match the benchmark (a few minutes)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           f'torch=={tv}+{cu}', f'torchvision=={vv}+{cu}',
                           '--index-url', f'https://download.pytorch.org/whl/{cu}'])
    have, cudnn, gpus = probe()
    print('torch after install:', have)
assert have == want, f'Could not install torch {want}; stopping.'

if cudnn != cudnn_want:
    pkg = 'nvidia-cudnn-cu13' if cu.startswith('cu13') else 'nvidia-cudnn-cu12'
    spec = f'{pkg}=={cudnn_want // 10000}.{cudnn_want // 100 % 100}.*'
    print(f'cuDNN is {cudnn}, needed {cudnn_want} -> installing {spec}')
    subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', spec])
    have, cudnn, gpus = probe()
print('torch', have, '| cuDNN', cudnn, '(needed', cudnn_want, ') | GPU:', gpus)
assert 'T4' in gpus, 'Not a T4 GPU; switch the runtime/accelerator to T4.'
FORCE_ENV = cudnn != cudnn_want
if FORCE_ENV:
    print('WARNING: could not match cuDNN. Continuing with --force-env: torch and GPU match, '
          'only the cuDNN patch level differs; results are statistically equivalent but may not match digit for digit.')
else:
    print('Environment matches the benchmark.')

In [ ]:
# === Selftest (first time, ~2 minutes): it must report success ===
if RUN_SELFTEST:
    assert subprocess.call([sys.executable, 'run_exp.py', '--selftest', '--out', '/content/tmp_selftest']) == 0, 'SELFTEST failed'

In [ ]:
# === Run ===
import subprocess, sys
subprocess.call([sys.executable, 'run_exp.py', '--plan', '--cells', *CELLS, '--exp', *EXPS])
p = subprocess.Popen([sys.executable, '-u', 'run_exp.py', '--exp', *EXPS, '--cells', *CELLS,
                      '--out', RESULTS, '--data', DATA] + (['--force-env'] if FORCE_ENV else []),
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end='', flush=True)
rc = p.wait()
print('Exit code', rc, '(2 = environment/configuration error, see the messages above)')

In [ ]:
# === Progress, and packaging for download ===
import os, shutil, subprocess, sys
subprocess.call([sys.executable, 'run_exp.py', '--status', '--cells', *CELLS, '--exp', *EXPS, '--out', RESULTS])
if os.path.isdir(RESULTS):
    z = shutil.make_archive(f'{PROJECT}/results_exp_B_v4', 'zip', PROJECT, 'results_exp')
    print('Created', z)